In [162]:
import pandas as pd
import numpy as np
import io

In [163]:
# ==== File Paths ====
duelist_file = r"Input_Files/duelist 02.03.2026.xlsx"
duelist_main_file = r"Input_Files/Duelist 28th Feb, 2026.xlsx"
insurance = r"Input_Files/Fagun insurance.xlsx"
output_file = r"Output_Files/updated_duelist.xlsx"

insurance = pd.read_excel(insurance, dtype={"MainCode": str})

duelist = pd.read_excel(
    duelist_file,
    dtype={"MainCode": str, "AcCodeForChg": str, "Nominee": str},
    engine="openpyxl"
)

if duelist_main_file.endswith(".xlsb"):
    duelist_main = pd.read_excel(
        duelist_main_file,
        dtype={"MainCode": str, "AcCodeForChg": str, "Nominee": str},
        engine="pyxlsb"
    )
else:
    duelist_main = pd.read_excel(
        duelist_main_file,
        dtype={"MainCode": str, "AcCodeForChg": str, "Nominee": str},
        engine="openpyxl"
    )

print(len(duelist))
duelist.head(2)

27456


,BBB,AT,AcTypeDesc,CC,OldClientCode,ClientCode,OldAcNum,MainCode,Name,Address,...,ModelDetails,LotNumber,DownPayment,VehicleModel,VehicleNo,Region,Obligor,DistrictCode,IntRateOfChgAc,ExpiryDateOfChgAc
0,001,20,HP-3WH (TEMPO),01,NaN,00024235,NaN,00102000024235000002,BISHONATH MAHATO,SAKHUWA PRASAUNI SAKHUWA PRASAUNI PARSA,...,FOTON OMEGA STREAM,NaN,40% Downpayment,5 FOTON,NaN,NaN,NaN,025,17.0,17/01/2026
1,001,20,HP-3WH (TEMPO),01,~~~~~,~~~~~,~~,~~,~~GROUP TOTAL 1,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [164]:
def clean_columns(df):
    df.columns = df.columns.str.strip()
    return df

duelsit = clean_columns(duelist)
duelist_main = clean_columns(duelist_main)
insurance = clean_columns(insurance)

In [165]:
duelist = duelist.copy()
duelist = duelist[(duelist['ClientCode']!="~~~~~")]
duelist['AgeingDays'] = pd.to_numeric(duelist['AgeingDays'], errors='coerce')
duelist = duelist.sort_values(by='AgeingDays', ascending=False)
duelist['AgeingDays'] = duelist['AgeingDays'].fillna("")
duelist.head(2)

,BBB,AT,AcTypeDesc,CC,OldClientCode,ClientCode,OldAcNum,MainCode,Name,Address,...,ModelDetails,LotNumber,DownPayment,VehicleModel,VehicleNo,Region,Obligor,DistrictCode,IntRateOfChgAc,ExpiryDateOfChgAc
18919,001,32,HP - HEAVY EQUIPMENT,01,NaN,00005629,NaN,00103200005629000002,AAYON NIRMAN SEWA PVT.LTD,"GHORAHI-10,DANG GHORAHI-10,DANG",...,JCB JS 220 TRACKED EXCAVATOR,NaN,40% Downpayment,7 HEAVY MACHINE,NaN,MID WESTERN,NaN,053,0.0,03/07/2025
19282,001,32,HP - HEAVY EQUIPMENT,01,001817,00003833,82001817041,00103200003833000004,RIDIPA CONSTRUCTION,Tewang-8 rolpa Rolpa District RAPTI ZONE,...,JCB JS205 LC Tracked Excavator,NaN,25% Downpayment,NaN,Ra 1 Ka 465,MID-WESTERN REGION,NaN,053,0.0,21/03/2025


In [166]:
pos = duelist.columns.get_loc("Name") + 1
duelist.insert(pos, "Ageing", np.nan)
duelist.insert(pos, "Loan Type", np.nan)
duelist.insert(pos, "OfficerName", np.nan)

pos = duelist.columns.get_loc("TotCharge") + 1
duelist.insert(pos, "OvDueWithInsurance", np.nan)
duelist.insert(pos, "InsurancePremium", np.nan)
duelist.insert(pos, "TotOvDue", np.nan)

pos = duelist.columns.get_loc("AgeingDays") + 1
duelist.insert(pos, "Bucket", np.nan)

pos = duelist.columns.get_loc("BranchName") + 1
duelist.insert(pos, "Dealer Name", np.nan)

In [167]:
# clean column names
duelist.columns = duelist.columns.str.strip()
duelist_main.columns = duelist_main.columns.str.strip()

# clean MainCode
duelist["MainCode"] = duelist["MainCode"].astype(str).str.strip()
duelist_main["MainCode"] = duelist_main["MainCode"].astype(str).str.strip()

cols = ["OfficerName", "Loan Type", "Dealer Name"]

df = duelist.merge(
    duelist_main[["MainCode"] + cols],
    on="MainCode",
    how="left",
    suffixes=("", "_ref")
)

for col in cols:
    df[col] = df[col].fillna(df[col + "_ref"])

df.drop(columns=[c + "_ref" for c in cols], inplace=True)

In [168]:
df.head(2)

,BBB,AT,AcTypeDesc,CC,OldClientCode,ClientCode,OldAcNum,MainCode,Name,OfficerName,...,ModelDetails,LotNumber,DownPayment,VehicleModel,VehicleNo,Region,Obligor,DistrictCode,IntRateOfChgAc,ExpiryDateOfChgAc
0,001,32,HP - HEAVY EQUIPMENT,01,NaN,00005629,NaN,00103200005629000002,AAYON NIRMAN SEWA PVT.LTD,Dang-Chronic Account-Mukesh Dahal-JCB,...,JCB JS 220 TRACKED EXCAVATOR,NaN,40% Downpayment,7 HEAVY MACHINE,NaN,MID WESTERN,NaN,053,0.0,03/07/2025
1,001,32,HP - HEAVY EQUIPMENT,01,001817,00003833,82001817041,00103200003833000004,RIDIPA CONSTRUCTION,Kathmandu-Chronic Account-Mukesh Dahal-JCB,...,JCB JS205 LC Tracked Excavator,NaN,25% Downpayment,NaN,Ra 1 Ka 465,MID-WESTERN REGION,NaN,053,0.0,21/03/2025


In [ ]:
# rename df1 columns to match df2
insurance = insurance.rename(columns={
    "prem" : "InsurancePremium"
})

# merge
df = df.merge(
    insurance[["MainCode", "InsurancePremium"]],
    on="MainCode",
    how="left",
    suffixes=("", "_ref")
)
# fill missing values
df["InsurancePremium"] = df["InsurancePremium"].fillna(df["InsurancePremium_ref"])

# remove extra columns
df.drop(columns=["InsurancePremium_ref"], inplace=True)

# fill remaining empty with 0
df["InsurancePremium"] = df["InsurancePremium"].fillna(0)

In [170]:
df.head(2)

,BBB,AT,AcTypeDesc,CC,OldClientCode,ClientCode,OldAcNum,MainCode,Name,OfficerName,...,ModelDetails,LotNumber,DownPayment,VehicleModel,VehicleNo,Region,Obligor,DistrictCode,IntRateOfChgAc,ExpiryDateOfChgAc
0,001,32,HP - HEAVY EQUIPMENT,01,NaN,00005629,NaN,00103200005629000002,AAYON NIRMAN SEWA PVT.LTD,Dang-Chronic Account-Mukesh Dahal-JCB,...,JCB JS 220 TRACKED EXCAVATOR,NaN,40% Downpayment,7 HEAVY MACHINE,NaN,MID WESTERN,NaN,053,0.0,03/07/2025
1,001,32,HP - HEAVY EQUIPMENT,01,001817,00003833,82001817041,00103200003833000004,RIDIPA CONSTRUCTION,Kathmandu-Chronic Account-Mukesh Dahal-JCB,...,JCB JS205 LC Tracked Excavator,NaN,25% Downpayment,NaN,Ra 1 Ka 465,MID-WESTERN REGION,NaN,053,0.0,21/03/2025


In [171]:
df.to_excel(output_file, index=False)